In [ ]:
1+1

In [ ]:
import tomllib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from astropy import units as u
from astropy.table import Table, hstack
import astropy

In [ ]:
from astropy import constants as c

In [ ]:
from sparcl.client import SparclClient

In [ ]:
from pathlib import Path

In [ ]:
import tomli_w

In [ ]:
from scipy.special import erf

In [ ]:
import arya
arya.init("mnras")
plt.rcParams["figure.figsize"] = (8, 3)

In [ ]:
C_KMS = c.c.to("km/s").value

In [ ]:
from scipy.ndimage import gaussian_filter1d

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import desi_utils
from desi_utils.plotting import plot_wrapped_spectrum
from desi_utils.spectral_lines import to_elsm_format
from desi_utils.fit_continuum import redshift_to_vel, vel_to_redshift
from desi_utils.catalogue_helpers import to_mel_line, to_fsf_line
from desi_utils import EMISSION_LINES, get_wavelength
from desi_utils import LineFitter
from desi_utils.fit_lines import _gaussian_flux

In [ ]:
import h5py

In [ ]:
def write_spec(spec, file):
    with h5py.File(file, "w") as f:
        f.create_dataset("wavelength", data=spec.wavelength)
        f.create_dataset("flux", data=spec.flux)
        f.create_dataset("uncertainty", data=spec.uncertainty)
        f.create_dataset("wave_dispersion", data=spec.wave_dispersion)

        for k, v in spec.meta.items():
            f.attrs[k] = v

## Data loading

In [ ]:
dwarf_main = Table.read("../data/desi_dr1_dwarf_catalog.fits", hdu=1)
dwarf_fsf = Table.read("../data/desi_dr1_dwarf_catalog.fits", hdu=4)

In [ ]:
dwarf_cat = hstack([dwarf_main, dwarf_fsf])

In [ ]:
fsf_vac_cat1 = Table.read("../data/fastspecfit_dwarfs_part1.fits")
fsf_vac_cat2 = Table.read("../data/fastspecfit_dwarfs_part2.fits")

In [ ]:
fsf_vac_cat = hstack((fsf_vac_cat1, fsf_vac_cat2))

In [ ]:
mel_cat = Table.read("../data/stellar_mass_emline_dwarfs.fits")

In [ ]:
# np.all(np.isin(fsf_cat["targetid"], dwarf_cat["targetid"]))

In [ ]:
with open("../analysis/selected_targetid.txt", "rb") as f:
    targetids = tomllib.load(f)["targetid"]

In [ ]:
skyspec_table = Table.read("../data/skyspectrum.fits")
skyspec = desi_utils.Spectrum(wavelength=skyspec_table["wavelength"], flux=skyspec_table["flux"], uncertainty=0)

In [ ]:
client = SparclClient()

# Restart here

In [ ]:
filt = np.isin(mel_cat["targetid"], dwarf_cat["TARGETID_1"])

In [ ]:
_targids = mel_cat["targetid"][filt]

In [ ]:
idx_highsnr = _targids[np.argsort(mel_cat["snr_med"][filt])][::-1]

In [ ]:
np.sum(np.isin(targetids, mel_cat["targetid"]))

In [ ]:
np.isin(targetids, mel_cat["targetid"])

In [ ]:
idx_in = 3

In [ ]:
targetid = targetids[idx_in]

In [ ]:
idx = np.where(dwarf_cat["TARGETID_1"] == targetid)[0][0]

In [ ]:
idx

In [ ]:
targetid

In [ ]:
targetid in mel_cat["targetid"]

In [ ]:
targetid in [int(i) for i in fsf_vac_cat["targetid"] ]

In [ ]:
out_dir = Path(".", "example_fits", str(targetid))

In [ ]:
if not out_dir.exists():
    out_dir.mkdir()

In [ ]:
galspec_raw = desi_utils.get_data.get_spectra(targetid)

In [ ]:
targetid

In [ ]:
if len(galspec_raw.flux.shape) == 2:
    galspec_raw= galspec_raw[0]
    for key in galspec_raw.meta.keys():
        if key in ["original_wcs"]:
            continue
        galspec_raw.meta[key] = galspec_raw.meta[key][0]

    

In [ ]:
spec = desi_utils.Spectrum.from_desi(galspec_raw)

In [ ]:

if not (out_dir / "spectrum.hdf5").exists():
    write_spec(spec, out_dir / "spectrum.hdf5")

In [ ]:
dwarf_props = dwarf_cat[idx]
dwarf_props

### The ELSM fit results

In [ ]:
mel_props = mel_cat[mel_cat["targetid"] == targetid][0]

In [ ]:
mel_fluxes = {}
for line in desi_utils.EMISSION_LINES:
    mel_line = to_mel_line(line)

    if mel_line is None:
        continue
    mel_fluxes[line + "_flux"] = mel_props[mel_line + "_flux"]
    mel_fluxes[line + "_flux_err"] = mel_props[mel_line + "_fluxerr"]
    mel_fluxes[line + "_sigma"] = mel_props[mel_line + "_sigma"]
    mel_fluxes[line + "_cen"] = mel_props[mel_line + "_center"]
    mel_fluxes[line + "_cont"] = mel_props[mel_line + "_cont"]

mel_fluxes["flux_scale"] = mel_props["flux_scale"]
mel_fluxes["redshift"] = mel_props["z"]

## The FSF fluxes

In [ ]:
targetid

In [ ]:
np.sum(fsf_vac_cat["targetid"] == targetid)

In [ ]:
fsf_vac_props = fsf_vac_cat[fsf_vac_cat["targetid"] == targetid][0]

In [ ]:
fsf_vac_fluxes = {}
v0 = redshift_to_vel(fsf_vac_props["Z"])

for line in desi_utils.EMISSION_LINES:
    fsf_line = to_fsf_line(line)

    if fsf_line is None:
        continue
    fsf_vac_fluxes[line + "_flux"] = fsf_vac_props[fsf_line + "_FLUX"]
    fsf_vac_fluxes[line + "_flux_err"] = fsf_vac_props[fsf_line + "_FLUX_IVAR"]**-0.5

    v = fsf_vac_props[fsf_line + "_VSHIFT"]
    z = vel_to_redshift(v)
    lambda0 = get_wavelength(line) * (1 + z)

    fsf_vac_fluxes[line + "_sigma"] = fsf_vac_props[fsf_line + "_SIGMA"] / C_KMS * lambda0

    fsf_vac_fluxes[line + "_cen"] = lambda0
    fsf_vac_fluxes[line + "_cont"] = fsf_vac_props[fsf_line + "_CONT"]

fsf_vac_fluxes["redshift"] = fsf_vac_props["Z"]

In [ ]:
fsf_fluxes = {}
v0 = redshift_to_vel(dwarf_props["Z"])

for line in desi_utils.EMISSION_LINES:
    fsf_line = to_fsf_line(line)

    if fsf_line is None:
        continue
    fsf_fluxes[line + "_flux"] = dwarf_props[fsf_line + "_FLUX"]
    fsf_fluxes[line + "_flux_err"] = dwarf_props[fsf_line + "_FLUX_IVAR"]**-0.5

    v = dwarf_props[fsf_line + "_VSHIFT"]
    z = vel_to_redshift(v)
    lambda0 = get_wavelength(line) * (1 + z)

    fsf_fluxes[line + "_sigma"] = dwarf_props[fsf_line + "_SIGMA"] / C_KMS * lambda0

    fsf_fluxes[line + "_cen"] = lambda0
    fsf_fluxes[line + "_cont"] = dwarf_props[fsf_line + "_CONT"]

fsf_fluxes["redshift"] = dwarf_props["Z"]

In [ ]:
# fsf_fluxes = {}
# v0 = redshift_to_vel(fsf_props["Z"])
# for line in desi_utils.EMISSION_LINES:
#     fsf_line = to_fsf_format(line)

#     if fsf_line is None:
#         continue
#     fsf_fluxes[line + "_flux"] = fsf_props[fsf_line + "_FLUX"]
#     fsf_fluxes[line + "_flux_err"] = fsf_props[fsf_line + "_FLUX_IVAR"]**0.5

#     v = fsf_props[fsf_line + "_VSHIFT"]
#     z = vel_to_redshift(v)
#     lambda0 = get_wavelength(line) * (1 + z)

#     fsf_fluxes[line + "_sigma"] = fsf_props[fsf_line + "_SIGMA"] / C_KMS * lambda0

#     fsf_fluxes[line + "_cen"] = lambda0
#     fsf_fluxes[line + "_cont"] = fsf_props[fsf_line + "_CONT"]

# fsf_fluxes["redshift"] = fsf_props["Z"]

# The spectrum as is

In [ ]:
np.median(dwarf_props["SNR_R"])

In [ ]:
np.max(dwarf_cat["SNR_R"])

In [ ]:
np.median(dwarf_cat["SNR_R"][np.isfinite(dwarf_cat["SNR_R"])])

In [ ]:
specdict = {
    "observations": spec,
    # "smoothed spectrum": (spec_resamp.wavelength, convolve(spec_resamp.flux, smoothing_kernel)),
    "sky": (skyspec.wavelength, 0.1*skyspec.flux) ,
    "std": (spec.wavelength, spec.uncertainty),
}
plot_wrapped_spectrum(
    {"observations": spec},
    redshift=spec.redshift, label_lines=desi_utils.EMISSION_LINES, show_uncertainty=True,
)

plt.savefig(out_dir / "obs_spec.png")

In [ ]:
smooth_spec = desi_utils.spectrum.smooth_spectrum(spec)

In [ ]:
plot_wrapped_spectrum(
    {"observations": spec,
     "sky/10": (skyspec.wavelength, skyspec.flux / 10),
    },
    {},
    redshift=spec.redshift, label_lines=desi_utils.EMISSION_LINES, show_uncertainty=True,
)

plt.savefig(out_dir / "obs_spec_w_sky.png")

In [ ]:
new_lines = [
    "H1r_3835A", #??
    "He1r_3889A", #??
    "H1r_3889A",
    "Ar4_4711A",
    "Ar4_4740A",
    "Fe2_5016A", #??
    "O1_6364A",
    # "H1r_8665A",8752, 8862, 9015, 9229
    # 6680>>>
]
    

### resampling sanity checks

In [ ]:
spec_resamp, flux_scale = desi_utils.spectrum.log_rebin_spectrum(spec)

In [ ]:
sps = desi_utils.fit_continuum.load_sps("spectra_fsps_9.0.npz", spec_resamp)

In [ ]:


plt.figure(figsize = (8, 2))

# Plot the original spectrum in maroon color
# plot_spectrum(smoothspec)

s = spec * flux_scale**-1

s.plot(lw=0.3)# plt.plot(galspec.spectral_axis, model)
spec_resamp.plot()
plt.yscale("asinh")
plt.xlabel(r'$\lambda$ [$\AA$] (observed)')
plt.show()



In [ ]:
plt.figure(figsize = (8, 2))

plt.plot(galspec_raw.spectral_axis, 1 / np.sqrt(galspec_raw.uncertainty.array) *flux_scale**-1)
plt.plot(spec.wavelength, spec.uncertainty * flux_scale**-1)
plt.plot(spec_resamp.wavelength, spec_resamp.uncertainty)
plt.yscale("asinh")
plt.xlabel(r'$\lambda$ [$\AA$] (observed)')
plt.ylabel("uncertainty")

plt.show()

In [ ]:
plt.figure(figsize = (8, 2))

plt.plot(galspec_raw.wavelength, galspec_raw.meta["wave_sigma"])
plt.plot(spec_resamp.wavelength, spec_resamp.wave_dispersion)
plt.yscale("asinh")
plt.xlabel(r'$\lambda$ [$\AA$] (observed)')
plt.ylabel("wave dispersion / $\\AA$")
plt.show()

# Continuum fitting

In [ ]:
sedmodel = desi_utils.fit_continuum.SEDModel(
    sps_file="spectra_fsps_9.0.npz"
)

In [ ]:
sedmodel

In [ ]:
ppxf_fit = sedmodel.fit(spec_resamp)

In [ ]:
ppxf_fit.plot()

In [ ]:
def plot_weight_fractions(ppxf_fit, ppxf_model, spec):
    sps = desi_utils.fit_continuum.load_sps(ppxf_model.sps_file, spec)
    light_weights = ppxf_fit.weights[~ppxf_fit.gas_component]            # Exclude weights of the gas templates
    light_weights = light_weights.reshape(ppxf_fit.reg_dim)        # Reshape to (n_ages, n_metal)

    sps.plot(light_weights/light_weights.sum())  # Normalize to light fractions


In [ ]:
plot_weight_fractions(ppxf_fit, sedmodel, spec_resamp)

In [ ]:
specdict = {
    "observations": spec_resamp,
    # "smooth_spec": smooth_spec,
    "total_fit": (spec_resamp.wavelength, ppxf_fit.bestfit),
    # "my_fit": (spec_resamp.wavelength, ppxf_fit_me.bestfit),
}
styles = {
    "observations": dict(alpha=1, lw=0.3, color="k", ),
}

plot_wrapped_spectrum(specdict, styles, show_uncertainty=True, label_lines=desi_utils.EMISSION_LINES, 
                     redshift = spec.redshift)


In [ ]:
[vel_to_redshift(s[0]) for s in ppxf_fit.sol]

In [ ]:
ppxf_fluxes = {}
for i in range(len(ppxf_fit.gas_names)):
    line = ppxf_fit.gas_names[i]
    flux = ppxf_fit.gas_flux[i]
    fluxerr = ppxf_fit.gas_flux_error[i]

    comp = ppxf_fit.component[ppxf_fit.gas_component][i]
    v_0, sigma_v = ppxf_fit.sol[comp]

    z = vel_to_redshift(v_0)

    # we have to correctly calculate the fluxes
    lam = ppxf_fit.lam
    dlog_lam = np.diff(np.log(lam))
    assert np.all(np.isclose(dlog_lam, dlog_lam[0]))
    dlog_lam = dlog_lam[0]
    f_lam = ppxf_fit.gas_bestfit_templates[:, i]
    
    flux_corrected = np.sum(f_lam * lam * dlog_lam) * flux_scale
    ppxf_fluxes[line + "_flux"] = flux_corrected
    ppxf_fluxes[line + "_flux_err"] = flux_corrected * fluxerr / flux

    lambda0 = get_wavelength(line) * (1 + z)
    ppxf_fluxes[line + "_cen"] = lambda0
    ppxf_fluxes[line + "_sigma"] = sigma_v / C_KMS * lambda0


    print(f"{line:16}{flux_corrected:8.2f}  pm{fluxerr*flux_scale:8.2f}", end="\t")

    # if line in elsm_fluxes.keys():
    #     print(f"{flux_corrected / elsm_fluxes[line]:8.2f}")
    # else:
    #     print()
        


In [ ]:
print(f"Desired Delta Chi^2: {np.sqrt(2*spec_resamp.flux.size)/ppxf_fit.dof:#.4g}")
print(f"Current Delta Chi^2: {(ppxf_fit.chi2 - 1)*spec_resamp.flux.size / ppxf_fit.dof:#.4g}")

### Spectral fit and residuals

In [ ]:
stars_bestfit = ppxf_fit.bestfit - ppxf_fit.gas_bestfit

In [ ]:
residual = spec_resamp.flux - ppxf_fit.bestfit

In [ ]:
specdict = {
    # "smoothed spectrum": (spec_resamp.wavelength, convolve(spec_resamp.flux, smoothing_kernel)),
    "stars fit": (spec_resamp.wavelength / (1 + spec.redshift), stars_bestfit),
    "total_fit": (spec_resamp.wavelength / (1 + spec.redshift), ppxf_fit.bestfit),
    "observations": (spec_resamp.wavelength / (1 + spec.redshift), spec_resamp.flux, spec_resamp.uncertainty),
}

In [ ]:
styles = {
    "observations": dict(alpha=1, lw=0.3, color="k", ),
}

In [ ]:
plot_wrapped_spectrum(specdict, styles, ylim=(-0.1, np.max(spec_resamp.flux) * 1.1), 
                      label_lines=desi_utils.EMISSION_LINES,
                      show_uncertainty=True,
                     redshift=0)

plt.savefig("ppxf_fit_spec.png")

In [ ]:
out_dir

In [ ]:
styles = {
    "residual": dict(alpha=1, color="k"),
}

spectra = {
    "residual": (spec_resamp.wavelength, residual, spec_resamp.uncertainty),
}
plot_wrapped_spectrum(spectra, styles, ylim=(-5, 5), show_uncertainty=True, label_lines=desi_utils.EMISSION_LINES, 
                     redshift = spec.redshift)


plt.savefig(out_dir / "ppxf_residual_spec.png")

In [ ]:
fig, ax = plt.subplots()
plt.hist(1.2*residual / spec_resamp.uncertainty, bins=np.linspace(-5, 5, 100), density=True)

x = np.linspace(-5, 5, 1000)
y = 1/np.sqrt(2*np.pi) * np.exp(-x**2 / 2)
plt.plot(x, y)

ax.set(
    xlim = (-5, 5),
    xlabel = "residual $z$-score",
    ylabel = "normalized density"
)

plt.savefig(out_dir / "zscore_hist.png")

In [ ]:
styles = {
    "residual": dict(alpha=1, lw=1.0, markersize=3, marker="."),
}

spectra = {
    "residual": ((spec_resamp.wavelength), residual / spec_resamp.uncertainty),
    "zeros": ((spec_resamp.wavelength), np.zeros_like(spec_resamp.wavelength)),

}

plot_wrapped_spectrum(spectra, styles, xlabel="wavelength", ylabel="z score", ylim=(-5, 5),
label_lines=desi_utils.EMISSION_LINES,
                     redshift=spec.redshift,
                     )

plt.savefig(out_dir / "zscore_residual_spec.png")

# Finding & fitting lines

## Comparing with catalogue fits

In [ ]:
multiplits = [
    ("O2_3726A", "O2_3729A"),
    ("Ne3_3968A", "H1r_3970A"),
    ("H1r_4341A", "O3_4363A"),
    ("He2r_4686A", "Ar4_4711A", "Ar4_4740A"),
    ("O3_4959A", "O3_5007A"),
    ("O1_6300A", "S3_6312A", "O1_6364A"), 
    ('N2_6548A', 'H1r_6563A', 'N2_6584A'),
    ("S2_6716A", "S2_6731A"),
    ("He1r_7281A", "O2_7320A", "O2_7331A"),
]

In [ ]:
all_multiplit_lines = np.concat(multiplits)

In [ ]:
all_multiplit_lines

In [ ]:
single_lines = [line for line in desi_utils.EMISSION_LINES if line not in all_multiplit_lines]

In [ ]:
single_lines

In [ ]:
assert len(single_lines) + len(all_multiplit_lines) == len(desi_utils.EMISSION_LINES)

In [ ]:
wave_range = (spec.get_wavelength(restframe=True).min(), spec.get_wavelength(restframe=True).max())

In [ ]:
def extract_region(lines, width = 100, wavelength_range=wave_range):
    if isinstance(lines, str):
        line = lines
        lambda0 = get_wavelength(line)
        if lambda0 < wavelength_range[0] or lambda0 > wavelength_range[1]:
            return None
        region = get_wavelength(line) - width / 2, get_wavelength(line) + width / 2

    else:
        lambdas = [get_wavelength(line) for line in lines]
        good_lambdas = [wavelength_range[0] < lam < wavelength_range[1] for lam in lambdas]
        lambdas = np.array(lambdas)[good_lambdas]
        
        region = np.min(lambdas) - width/2, np.max(lambdas) + width / 2

    return region

In [ ]:
single_regions = [extract_region(line) for line in single_lines]

In [ ]:
multi_regions = [extract_region(line) for line in multiplits]

In [ ]:
single_regions

In [ ]:
multi_regions

In [ ]:
def plot_regions(regions):
    for region in regions:
        if region is None:
            continue
        x1, x2 = region
        plt.plot([x1, x1, x2, x2], [0, 1, 1, 0])
        

In [ ]:
spec.plot(restframe=True)
plot_regions(single_regions)
plot_regions(multi_regions)

In [ ]:
if not (out_dir / "lines").exists():
    (out_dir / "lines").mkdir()

In [ ]:
flat_spec = desi_utils.Spectrum(spec_resamp.wavelength, (spec_resamp.flux - stars_bestfit) * flux_scale, 
                     uncertainty=spec_resamp.uncertainty * flux_scale, redshift=spec_resamp.redshift)

In [ ]:
LINE_WAVES = np.array([desi_utils.get_wavelength(l) for l in desi_utils.EMISSION_LINES])

def fit_region(spec, region):
    line_filt = LINE_WAVES > region[0]
    line_filt &= LINE_WAVES < region[1]
    line_names = np.array(desi_utils.EMISSION_LINES)[line_filt]
    print("fitting lines: ", line_names)
    
    subspec = desi_utils.spectrum.clip_spectrum(flat_spec, region)
    line_fitter = LineFitter(line_names = line_names, progressbar=False)
    fit = line_fitter.fit(subspec)    

    return fit
    

In [ ]:
all_regions = [*single_regions, *multi_regions]

# The main loop: fitting all regions

In [ ]:
region_fits = []
for region in all_regions:
    if region is None:
        continue
    fit = fit_region(spec, region)
    region_fits.append(fit)

In [ ]:

def plot_fit(fit):
    for line in fit.line_names:
        fig, ax = plt.subplots(1)
        
        fit.plot_line_fit(line, wave_range=10, ax=ax, show_legend=False)
        ax.set(title = desi_utils.spectral_lines.format_line(line))
        
        x0 = fit.line_centers[fit.line_names.index(line)]
        filt = spec_resamp.wavelength > x0 - 10
        filt &= spec_resamp.wavelength < x0 + 10
        x_model = spec_resamp.wavelength[filt]
        x_edges = tuple(e[:, None] for e in desi_utils.fit_lines._pixel_edges(x_model))


        df = fit.summary()["median"]
        
        mu, sigma, A = df[f"mu[{line}]"], df[f"sigma[{line}]"], df[f"amp[{line}]"]
        
        print(line)
        flux = A * np.sqrt(2*np.pi) * sigma

        sigma_lam =  spec_resamp.wave_dispersion[np.argmin(np.abs(spec.wavelength - x0))]


        plot_gaussian_from_dict(x_model, fsf_fluxes, line, label="FSF", dispersion=sigma_lam, color="C1", flux_scale = 1)
        plot_gaussian_from_dict(x_model, fsf_vac_fluxes, line, label="FSF-VAC", dispersion=sigma_lam, color="C3", ls="--",  flux_scale = 1)
        plot_gaussian_from_dict(x_model, mel_fluxes, line, label="MEL", flux_scale=1, color="C2")


        ax.legend()
        plt.savefig(out_dir / f"lines/{line}.png")


In [ ]:
def plot_gaussian(x_model, mu, sigma, flux, cont, dispersion=None, **kwargs):
    amp = flux / sigma / np.sqrt(2*np.pi)
    lam0 = mu * (1 + spec.redshift)
    
    x_edges = tuple(e[:, None] for e in desi_utils.fit_lines._pixel_edges(x_model))

    y_model = _gaussian_flux(
        x_model, amp, 
        lam0, sigma,
        edges = x_edges, 
        exp = np.exp, 
        erf = erf
    )

    if dispersion is not None:
        y_model = gaussian_filter1d(y_model, dispersion / np.median(np.diff(x_model)))


    y_model += cont
    plt.plot(x_model, y_model, **kwargs)


In [ ]:
def plot_gaussian_from_dict(x_model, flux_dict, line, dispersion=None, flux_scale=1, **kwargs):
    if line + "_flux" not in flux_dict.keys():
        return
    mu = flux_dict[line + "_cen"]
    sigma = flux_dict[line + "_sigma"]
    flux = flux_dict[line + "_flux"] * flux_scale
    cont = flux_dict[line + "_cont"] * flux_scale

    amp = flux / sigma / np.sqrt(2*np.pi)
    lam0 = mu * (1 + spec.redshift)
    
    plot_gaussian(x_model, mu, sigma, flux, cont, dispersion=dispersion,  **kwargs)

In [ ]:
def plot_spec_subregion(spec, xlim):
    filt = spec.wavelength > xlim[0]
    filt &= spec.wavelength < xlim[1]

    plt.errorbar(spec.wavelength[filt], spec.flux[filt], yerr=spec.uncertainty[filt], fmt="ko", capsize=0)

    return spec.wavelength[filt]

In [ ]:
flux_unit = r"$10^{-17}$ erg s$^{-1}$ cm$^{-2}$"

In [ ]:
region_fits[1]._means()["amp"][0][0]

In [ ]:
def plot_real_fit(fit):
    for line in fit.line_names:
        fig, ax = plt.subplots(1)        
        x0 = fit.line_centers[fit.line_names.index(line)]


        xlim = x0-10, x0+10
        x_model = plot_spec_subregion(spec, xlim)
        sigma_lam =  spec.wave_dispersion[np.argmin(np.abs(spec.wavelength - x0))]
        filt = (spec_resamp.wavelength > xlim[0]) & (spec_resamp.wavelength < xlim[1])
        cont = np.median(stars_bestfit[filt])

        bestfit = fit._means()
        idx = fit.line_names.index(line)
        
        bestfit_amp = bestfit["amp"][idx][0]
        bestfit_mu = bestfit["mu"][idx][0] / (1 + spec.redshift)
        bestfit_sigma = bestfit["sigma"][idx][0]
        bestfit_flux = bestfit_amp * bestfit_sigma * np.sqrt(2*np.pi)
        plot_gaussian(x_model, bestfit_mu, bestfit_sigma, bestfit_flux, cont, color="C0", label="MCMC")



        # mel_scale = cont / elsm_fluxes.get(line + "_cont", 1)
        # print(line, "\t", f"{mel_scale:0.3g}")
        sigma_lam =  spec.wave_dispersion[np.argmin(np.abs(spec.wavelength - x0))]
        plot_gaussian_from_dict(x_model, fsf_fluxes, line, label="FSF", dispersion=sigma_lam, color="C1", flux_scale = 1)
        plot_gaussian_from_dict(x_model, fsf_vac_fluxes, line, label="FSF-VAC", dispersion=sigma_lam, color="C3", ls="--", flux_scale = 1)
        plot_gaussian_from_dict(x_model, mel_fluxes, line, label="MEL", flux_scale=1, color="C2")
        
        ax.set(
            title = desi_utils.spectral_lines.format_line(line),
            xlabel = "observed wavelength / $\\AA$",
            ylabel = f"flux / {flux_unit}",
            ylim = 0,
            xlim = xlim
        )
        ax.legend()
        plt.savefig(out_dir / f"lines/{line}_observed.png")


In [ ]:
flux_corr = np.median(spec.flux)

In [ ]:
for fit in region_fits:
    plot_real_fit(fit)

In [ ]:
for fit in region_fits:
    plot_fit(fit)

# Writing the results

In [ ]:
mc_fluxes = {}
for fit in region_fits:
    for line in fit.line_names:
        df = fit.summary()["median"]
        df_err = fit.summary()["mad"]
        
        flux = fit.line_fluxes()["median"][f"line_flux[{line}]"]
        flux_err = fit.line_fluxes()["mad"][f"line_flux[{line}]"]
        
        mu = df[f"mu[{line}]"]
        mu_err = df_err[f"mu[{line}]"]
        
        
        
        sigma = df[f"sigma[{line}]"]
        sigma_err = df_err[f"sigma[{line}]"]

        mc_fluxes[line + "_flux"] = flux
        mc_fluxes[line + "_flux_err"] = flux_err
        mc_fluxes[line + "_sigma"] = sigma
        mc_fluxes[line + "_sigma_err"] = sigma_err
        mc_fluxes[line + "_cen"] = mu
        mc_fluxes[line + "_cen_err"] = mu_err

In [ ]:
mc_fluxes

In [ ]:
df_all = {
    "mcmc": mc_fluxes, 
    "ppxf": ppxf_fluxes,
    # "elsm": elsm_fluxes,
    "fsf": fsf_fluxes
}

In [ ]:
# with open(out_dir / "fluxes.toml", "wb") as f:
#     tomli_w.dump(df_all, f)

In [ ]:
df_ppxf = {
    "weights": list(ppxf_fit.weights),
    "gas_bestfit": list(ppxf_fit.gas_bestfit),
    "bestfit": list(ppxf_fit.bestfit),
    "sol": [list(l) for l in ppxf_fit.sol],
    "gas_names": list(ppxf_fit.gas_names),
    "chi2": ppxf_fit.chi2,
    "component": [int(i) for i in ppxf_fit.component],
    "gas_reddening": ppxf_fit.gas_reddening,
}
    

In [ ]:
with open(out_dir / "ppxf_fit.toml", "wb") as f:
    tomli_w.dump(df_ppxf, f)

# Overall comparison plot

In [ ]:
plt.errorbar

In [ ]:
def plot_correlation(mc_fluxes, elsm_fluxes, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()

    xmax = 0
    xmin = 0
    
    for line in EMISSION_LINES:
        x0 = mc_fluxes.get(line + "_flux", np.nan)
        xerr = mc_fluxes.get(line + "_flux_err", np.nan)
        y0 = elsm_fluxes.get(line + "_flux", np.nan)
        yerr = elsm_fluxes.get(line + "_flux_err", np.nan)
        
        if np.isfinite(xerr) and np.isfinite(yerr) and yerr < y0:
            ax.errorbar(x0, y0, yerr, xerr, fmt="ko", elinewidth=0.5, capsize=0)
            xmax = np.nanmax([x0, xmax, y0])
            xmin = np.nanmin([x0, xmin, y0])
            ax.annotate(line.replace("_", " "), (x0, y0), (5, 0),  fontsize=3, textcoords="offset points")

    ax.plot([xmin, xmax], [xmin, xmax])
    ax.set(
        aspect = 1,
        xlim=(1, xmax),
        ylim=(1, xmax),
        xscale = "log",
        yscale = "log", 
        **kwargs
    )

In [ ]:
def plot_resid(mc_fluxes, elsm_fluxes, ylabel, ax=None):
    if ax is None:
        ax = plt.gca()

    max_dev = 0
    
    for line in EMISSION_LINES:
        x0 = mc_fluxes.get(line + "_flux", np.nan)
        xerr = mc_fluxes.get(line + "_flux_err", np.nan)
        y0 = elsm_fluxes.get(line + "_flux", np.nan)
        yerr = elsm_fluxes.get(line + "_flux_err", np.nan)
        lam = mc_fluxes.get(line + "_cen", np.nan)
    
        tot_err =  (yerr/np.abs(y0) + xerr/np.abs(x0)) / np.abs(x0)
    
        if tot_err > 1:
            tot_err = 0
            color = "grey"
        else:
            color = "k"
        if np.isfinite(xerr) and np.isfinite(yerr) and (yerr < y0):
            ms = np.log10(x0)
            ax.errorbar(lam, np.log10(y0 / x0), tot_err, fmt="o", color=color, elinewidth=0.5, capsize=0, ms=ms)
            ax.annotate(line.replace("_", " "), (lam, np.log10(y0 / x0)), (5, 0),  fontsize=3, textcoords="offset points")

    
    ax.axhline(0)
    ax.set(
        xlabel = "wavelength",
        # ylim=(1 / 10**max_dev, 1 * 10**max_dev),
        ylabel = ylabel
    )

In [ ]:
def plot_both(mc_fluxes, mel_fluxes, xlabel, ylabel):
    fig, axs = plt.subplots(1, 2)

    plot_correlation(mc_fluxes, mel_fluxes, xlabel=xlabel, ylabel=ylabel, ax=axs[0])
    plot_resid(mc_fluxes, mel_fluxes, ylabel="log " + ylabel + "/" + xlabel, ax=axs[1])

In [ ]:
plot_both(mc_fluxes, mel_fluxes, xlabel="MC", ylabel="MEL")
plt.savefit(outdir / "MC_vs_MEL_fluxes.pdf")

In [ ]:
plot_both(mc_fluxes, fsf_fluxes, xlabel="MC", ylabel="FSF-dwarf")
plt.savefit(outdir / "MC_vs_fsf_dwarf_fluxes.pdf")

In [ ]:
plot_both(mc_fluxes, fsf_vac_fluxes, xlabel="MC", ylabel="FSF-VAC")
plt.savefit(outdir / "MC_vs_fsf_vac_fluxes.pdf")

In [ ]:
plot_both(fsf_fluxes, fsf_vac_fluxes, xlabel="FSF-dwarf", ylabel="FSF-VAC")
plt.savefit(outdir / "MC_vs_fsf_vac_fluxes.pdf")

In [ ]:
plot_both(fsf_fluxes, mel_fluxes, xlabel="FSF-dwarf", ylabel="MEL")